# Testing the hierarchical reserving models on a simulated portfolio (baseline scenario)
Author: Christophe Nozaradan (extension of a previous version by Jens Robben)  
Date: November 2021

## Introduction
This version of this document extends a previous one written by Jens Robben, in which hierarchical GLM and hierarchical GBM are compared in the case of an extreme scenario. In the present version, hierarchical XGB, hierarchical MLP and hierarchical CANN are further considered in the case of a baseline event scenario. 

## Simulating the portfolio
We evaluate the predictive performance of our hierarchical reserving model (hierarchical GLM, hierarchical GBM, hierarchical XGB and hierarchical MLP) on one portfolio simulated along the baseline event scenario. We compare the results with the ones obtained using an aggregate reserving model, namely the classical chain ladder model.

The seed number is:

In [1]:
seed = 1
print(seed)

1



First, we load the required packages.

In [9]:
import pandas as pd
import numpy as np
import hirempy as hrm

We simulate one portfolio along the baseline event scenario. The portfolio initially consists of 125 000
 claims, but only the claims that are reported between January 1, 2012 and December 31, 2020 are retained. Claims are tracked over 9 development years since reporting. In addition, we create an extra covariate devYearMonth, representing a simplified version of the interaction effect between dev.year and rep.month.

In [3]:
# The complete portfolio
reserving_data = hrm.simulate_scenario_baseline(seed = seed, n = 125000)

reserving_data.head(10)

,claim.nr,type,hidden,occ.date,occ.year,occ.year.fact,occ.month,rep.delay,rep.date,rep.year,rep.year.fact,rep.month,pay.nr,pay.date,pay.delay,pay.amount,dev.year,settlement.date
0,1,T1,M,2012-11-27,2012,2012,11,141.0,2013-04-17,2,2,4,1,2013-06-03,0.128679,146.906642,1,2013-09-01
0,1,T1,M,2012-11-27,2012,2012,11,141.0,2013-04-17,2,2,4,2,2013-07-07,0.221766,431.043692,1,2013-09-01
1,2,T2,M,2010-08-24,2010,2010,8,78.0,2010-11-10,-1,-1,11,1,2011-01-19,0.191650,828.059376,2,2012-12-16
1,2,T2,M,2010-08-24,2010,2010,8,78.0,2010-11-10,-1,-1,11,2,2011-06-13,0.588638,282.353961,2,2012-12-16
1,2,T2,M,2010-08-24,2010,2010,8,78.0,2010-11-10,-1,-1,11,3,2011-07-10,0.662560,482.025741,2,2012-12-16
1,2,T2,M,2010-08-24,2010,2010,8,78.0,2010-11-10,-1,-1,11,4,2011-08-08,0.741958,74.849930,2,2012-12-16
2,3,T2,M,2020-11-24,2020,2020,11,35.0,2020-12-29,9,9,12,1,2021-01-05,0.019165,181.332778,2,2027-09-23
2,3,T2,M,2020-11-24,2020,2020,11,35.0,2020-12-29,9,9,12,2,2021-07-01,0.503765,237.301332,2,2027-09-23
2,3,T2,M,2020-11-24,2020,2020,11,35.0,2020-12-29,9,9,12,3,2021-08-27,0.659822,1076.355658,2,2027-09-23
2,3,T2,M,2020-11-24,2020,2020,11,35.0,2020-12-29,9,9,12,4,2022-03-10,1.193703,131.942060,3,2027-09-23


In [4]:

# Creating the interaction effect
#reserving_data['monthDev12'] = reserving_data['rep.month'].astype(str)
#reserving_data.loc[reserving_data['dev.year'] > 3, 'monthDev12'] = 'dev.year.over.3'
#reserving_data['devYearMonth'] = reserving_data['dev.year'].astype(str) + '-' + reserving_data['monthDev12']

# Observed and prediction data set
observed_data = reserving_data[reserving_data['rep.date'] <= '2020-12-31']
prediction_data = reserving_data[reserving_data['rep.date'] > '2020-12-31']

In [5]:
observed_data.head(10)

,claim.nr,type,hidden,occ.date,occ.year,occ.year.fact,occ.month,rep.delay,rep.date,rep.year,rep.year.fact,rep.month,pay.nr,pay.date,pay.delay,pay.amount,dev.year,settlement.date
0,1,T1,M,2012-11-27,2012,2012,11,141.0,2013-04-17,2,2,4,1,2013-06-03,0.128679,146.906642,1,2013-09-01
0,1,T1,M,2012-11-27,2012,2012,11,141.0,2013-04-17,2,2,4,2,2013-07-07,0.221766,431.043692,1,2013-09-01
1,2,T2,M,2010-08-24,2010,2010,8,78.0,2010-11-10,-1,-1,11,1,2011-01-19,0.191650,828.059376,2,2012-12-16
1,2,T2,M,2010-08-24,2010,2010,8,78.0,2010-11-10,-1,-1,11,2,2011-06-13,0.588638,282.353961,2,2012-12-16
1,2,T2,M,2010-08-24,2010,2010,8,78.0,2010-11-10,-1,-1,11,3,2011-07-10,0.662560,482.025741,2,2012-12-16
1,2,T2,M,2010-08-24,2010,2010,8,78.0,2010-11-10,-1,-1,11,4,2011-08-08,0.741958,74.849930,2,2012-12-16
2,3,T2,M,2020-11-24,2020,2020,11,35.0,2020-12-29,9,9,12,1,2021-01-05,0.019165,181.332778,2,2027-09-23
2,3,T2,M,2020-11-24,2020,2020,11,35.0,2020-12-29,9,9,12,2,2021-07-01,0.503765,237.301332,2,2027-09-23
2,3,T2,M,2020-11-24,2020,2020,11,35.0,2020-12-29,9,9,12,3,2021-08-27,0.659822,1076.355658,2,2027-09-23
2,3,T2,M,2020-11-24,2020,2020,11,35.0,2020-12-29,9,9,12,4,2022-03-10,1.193703,131.942060,3,2027-09-23


In [6]:
prediction_data.head(10)

,claim.nr,type,hidden,occ.date,occ.year,occ.year.fact,occ.month,rep.delay,rep.date,rep.year,rep.year.fact,rep.month,pay.nr,pay.date,pay.delay,pay.amount,dev.year,settlement.date
27,28,T1,M,2020-12-11,2020,2020,12,80.0,2021-03-01,10,10,3,1,2021-03-23,0.060233,740.848627,1,2022-03-12
27,28,T1,M,2020-12-11,2020,2020,12,80.0,2021-03-01,10,10,3,2,2021-04-02,0.087611,47.518913,1,2022-03-12
27,28,T1,M,2020-12-11,2020,2020,12,80.0,2021-03-01,10,10,3,3,2021-07-16,0.375086,193.390905,1,2022-03-12
27,28,T1,M,2020-12-11,2020,2020,12,80.0,2021-03-01,10,10,3,4,2022-01-12,0.867899,183.772085,2,2022-03-12
27,28,T1,M,2020-12-11,2020,2020,12,80.0,2021-03-01,10,10,3,5,2022-02-10,0.947296,341.730430,2,2022-03-12
61,62,T3,L,2020-12-03,2020,2020,12,282.0,2021-09-11,10,10,9,1,2023-05-29,1.711157,161.390172,3,2030-10-10
61,62,T3,L,2020-12-03,2020,2020,12,282.0,2021-09-11,10,10,9,2,2024-01-25,2.370979,402.193058,4,2030-10-10
61,62,T3,L,2020-12-03,2020,2020,12,282.0,2021-09-11,10,10,9,3,2027-08-07,5.902806,19891.863563,7,2030-10-10
61,62,T3,L,2020-12-03,2020,2020,12,282.0,2021-09-11,10,10,9,4,2027-08-23,5.946612,16351.987812,7,2030-10-10
61,62,T3,L,2020-12-03,2020,2020,12,282.0,2021-09-11,10,10,9,5,2028-03-28,6.543463,5100.612247,8,2030-10-10


## Calibration of the hierarhical reserving models
We first define the weights used in the calibration of the hierarchical reserving model. The purpose of these weights is to replicate the development year distribution from the prediction data set to the observed data set (training data set). Hence, more weight is assigned to claims observations in later development years since reporting. We give a very small weight to the claims observations in the first development year since reporting because the prediction data set does not contain observations for this development year. More details are provided in @hirempaper

In [11]:
# Calculating the weights

# Filter and group the data
reported_claims = observed_data[observed_data['dev.year'] == 1]
reported_claims = reported_claims.groupby('rep.year').size()



# Calculate denominator and numerator
denominator = np.cumsum(reported_claims[::-1])[:-1][::-1]
numerator = np.cumsum(reported_claims[::-1])[1:][::-1]

print(denominator)
print(numerator)

# Calculate weight
#weight = np.concatenate(([10**(-6)], numerator / denominator))

# Create a named series
#weight_series = pd.Series(weight, index=[f'dev.year{i+1}' for i in range(9)])
print(weight_series)


#reported_claims <- observed_data %>%
#  dplyr::filter(dev.year == 1) %>%
#  group_by(rep.year) %>% 
#  dplyr::summarise(count = n()) %>%
#  pull(count)

#denominator <- tail(rev(cumsum(reported_claims)), -1)
#numerator <- head(cumsum(rev(reported_claims)), -1)
#weight <- c(10^(-6), numerator / denominator)

#names(weight) <- paste0('dev.year',1:9)
#weight

rep.year
0    147989
1    133244
2    118576
3    103912
4     89241
5     74368
6     59207
7     44593
8     29725
9     14897
dtype: int64
rep.year
-1    157612
 0    147989
 1    133244
 2    118576
 3    103912
 4     89241
 5     74368
 6     59207
 7     44593
 8     29725
dtype: int64


NameError: name 'weight_series' is not defined

'/home/nigel/hirempy/examples/baseline'